In [5]:
import pandas as pd

# Step 1: Load only needed columns
columns_to_use = ['LEAVE_TIME', 'STOP_TIME', 'GARAGE', 'ROUTE_NUMBER',
           'ESTIMATED_LOAD', 'ONS', 'OFFS', 'DWELL', 'SERVICE_DATE']
file2 = "/Users/kathypresto/Desktop/trimet/Big_Data/untitled folder 2/trimet_stop_event_2022_2023.csv"
df_2022 = pd.read_csv(file2, usecols=columns_to_use)

# Step 2: Filter out train garages
df_2022 = df_2022[~df_2022['GARAGE'].isin(['RUBY', 'ELMO'])]

# Step 3: Exclude non-standard routes (900s and above)
df_2022 = df_2022[df_2022['ROUTE_NUMBER'] < 900]

In [6]:
# calculate on-time flag
df_2022['DELAY_MINUTES'] = (df_2022['LEAVE_TIME'] - df_2022['STOP_TIME']) / 60
df_2022['ON_TIME'] = df_2022['DELAY_MINUTES'].between(-1, 5).astype(int)
df_2022['YEAR'] = pd.to_datetime(df_2022['SERVICE_DATE'], format='%d%b%Y:%H:%M:%S').dt.year

In [7]:

from scipy.stats import chi2_contingency

contingency_garage = pd.crosstab(df_2022['GARAGE'], df_2022['ON_TIME'])
chi2, p, dof, expected = chi2_contingency(contingency_garage)

print("Contingency Table:")
print(contingency_garage)
print("\nChi-Square Test Statistic:", chi2)
print("Degrees of Freedom:", dof)
print("P-Value:", p)



Contingency Table:
ON_TIME        0         1
GARAGE                    
CENTER   7517909  26913220
MERLO    6479020  21664066
POWELL   5832287  20104128

Chi-Square Test Statistic: 12700.379956307252
Degrees of Freedom: 2
P-Value: 0.0


In [8]:
# Group by Garage + Route + Year
route_summary_2022 = df_2022.groupby(['YEAR', 'GARAGE', 'ROUTE_NUMBER']).agg(
    Total_Trips=('ON_TIME', 'count'),
    On_Time_Trips=('ON_TIME', 'sum'),
    Avg_Dwell=('DWELL', 'mean'),
    Avg_Load=('ESTIMATED_LOAD', 'mean'),
    Avg_Ons=('ONS', 'mean'),
    Avg_Offs=('OFFS', 'mean')
).reset_index()

route_summary_2022['OTP_RATE'] = (route_summary_2022['On_Time_Trips'] / route_summary_2022['Total_Trips']) * 100

# Sort for bottom and top 10
bottom_routes = route_summary_2022.sort_values('OTP_RATE').head(10)
top_routes = route_summary_2022.sort_values('OTP_RATE', ascending=False).head(10)


In [9]:
print("🚨 Bottom 10 OTP Routes:")
print(bottom_routes)

print("\n🏆 Top 10 OTP Routes:")
print(top_routes)

🚨 Bottom 10 OTP Routes:
     YEAR  GARAGE  ROUTE_NUMBER  Total_Trips  On_Time_Trips  Avg_Dwell  \
32   2022  CENTER            55          210             88   2.476190   
167  2023   MERLO             4          819            415   5.769231   
37   2022  CENTER            66        82507          45933   7.249446   
208  2023  POWELL            87         2405           1342   4.926819   
164  2023  CENTER            79           63             36   2.380952   
100  2022   MERLO            96       305877         175734   5.027413   
98   2022   MERLO            92        37127          21453   4.824979   
144  2023  CENTER            12          723            427   6.800830   
133  2022  POWELL            98         5506           3256  35.886669   
18   2022  CENTER            22        21418          12685   3.366281   

     Avg_Load   Avg_Ons  Avg_Offs   OTP_RATE  
32   0.895238  0.033333  0.028571  41.904762  
167  4.610501  0.253968  0.253968  50.671551  
37   4.200298  0.477

In [10]:
routes_to_plot = [94, 96, 98, 66, 22, 23, 80, 87]

# Filtered DataFrame just for 2022
filtered_2022 = df_2022[df_2022['ROUTE_NUMBER'].isin(routes_to_plot)]

# Now check which garages show up
print(filtered_2022['GARAGE'].value_counts())


GARAGE
POWELL    1848538
MERLO     1246869
CENTER     145717
Name: count, dtype: int64


In [11]:
contingency = pd.crosstab(df_2022['GARAGE'], df_2022['ON_TIME'])
chi2, p, dof, expected = chi2_contingency(contingency)

print("Contingency Table:\n", contingency)
print("\nChi-Square Test Statistic:", chi2)
print("Degrees of Freedom:", dof)
print("P-Value:", p)

Contingency Table:
 ON_TIME        0         1
GARAGE                    
CENTER   7517909  26913220
MERLO    6479020  21664066
POWELL   5832287  20104128

Chi-Square Test Statistic: 12700.379956307252
Degrees of Freedom: 2
P-Value: 0.0


In [14]:
filtered_2022['GARAGE'].value_counts()

GARAGE
POWELL    1848538
MERLO     1246869
CENTER     145717
Name: count, dtype: int64

In [20]:
df_2022['YEAR'].unique()

array([2022, 2023], dtype=int32)